In [1]:

# AC-01 supplemental inspection only; no training, inference, or checkpoints.
import contextlib, hashlib, importlib.util, io, json, platform, tarfile
from datetime import datetime, timezone, timedelta
from pathlib import Path
from urllib.request import urlopen
from urllib.parse import quote
import torch, torchvision, yaml
from google.colab import files

COMMIT = '3faffbb8da00b773cc408a9c52a93ef5aa1c717f'
TRAINING_COMMIT = '735e5568f96819a9e52d25298ba5c34e213fa10b'
SESSION = 'colab-t4-20261113-browser-01'
OUT = Path('/content/model-architecture-colab-20261113')
assert not OUT.exists(), 'Evidence directory already exists; do not overwrite.'
OUT.mkdir()
def fetch(path):
    url = f'https://raw.githubusercontent.com/QuocAn118/Lab/{COMMIT}/' + quote('Lab 2/' + path)
    with urlopen(url, timeout=60) as response:
        return response.read()
source = fetch('src/lab_2/model.py')
source_hash = hashlib.sha256(source).hexdigest()
assert source_hash == '7fd9e8db551c25c22d12089c64c403fdb66ad99aae1a0f1d2ae34af018d657a6'
(OUT / 'model.py').write_bytes(source)
config = yaml.safe_load(fetch('configs/lab_2.yaml'))
expected = json.loads(fetch(f'reports/sessions/{SESSION}/reports/metrics.json'))['model']
spec = importlib.util.spec_from_file_location('audited_model', OUT / 'model.py')
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
counts, stream = {}, io.StringIO()
with contextlib.redirect_stdout(stream):
    print('Colab CPU architecture inspection; pretrained=False; no training performed.')
    print('Source commit:', COMMIT, 'Model SHA256:', source_hash)
    for key, name, unfreeze in [('resnet18','resnet18',[]), ('densenet121','densenet121',[]), ('resnet18_unfreeze_layer4','resnet18',['layer4'])]:
        model = module.build_model(name, config['model']['dropout'], unfreeze=unfreeze, pretrained=False)
        print('\nMODEL:', key)
        print(model)
        for parameter_name, parameter in model.named_parameters():
            print(parameter_name, tuple(parameter.shape), 'train' if parameter.requires_grad else 'frozen')
        counts[key] = module.count_parameters(model)
        assert counts[key] == expected[key], (key, counts[key], expected[key])
        print('COUNTS:', counts[key])
        del model
    print('PASS: all 3 model parameter counts match the completed T4 session.')
(OUT / 'architecture.txt').write_text(stream.getvalue(), encoding='utf-8')
created = datetime.now(timezone(timedelta(hours=7))).isoformat()
audit = dict(kind='architecture_inspection', created_at=created, operator='OpenAI Codex, authorized by HoangDang', source_commit=COMMIT, training_source_commit=TRAINING_COMMIT, training_session=SESSION, model_sha256=source_hash, training_performed=False, inference_performed=False, pretrained=False, device='cpu', python=platform.python_version(), torch=torch.__version__, torchvision=torchvision.__version__, model=counts, matches_t4_model_counts=True)
(OUT / 'audit.json').write_text(json.dumps(audit, indent=2)+'\n', encoding='utf-8')
(OUT / 'SOURCES.md').write_text(f'# Supplemental Colab architecture evidence\n\nLast updated: {created}\n\nCreated by: OpenAI Codex, authorized by HoangDang.\n\nOriginal model author: Xuan Bao (hbao2123-debug); package integration: HoangDang.\n\nRead-only CPU architecture inspection of model.py from QuocAn118/Lab commit {COMMIT}. SHA256 matches training source {TRAINING_COMMIT}. No training, inference, optimizer, pretrained weight download, or checkpoint write. All three counts match {SESSION}; this inspection supplies architecture/layer stdout missing from the saved T4 notebook and does not create a new training result.\n', encoding='utf-8')
manifest = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(OUT.iterdir()) if p.is_file()}
(OUT / 'FILES.json').write_text(json.dumps(manifest, indent=2)+'\n', encoding='utf-8')
archive = OUT.with_suffix('.tar.gz')
with tarfile.open(archive, 'w:gz') as handle:
    handle.add(OUT, arcname=OUT.name)
print(stream.getvalue())
print(json.dumps(audit, indent=2))
files.download(str(archive))

Colab CPU architecture inspection; pretrained=False; no training performed.
Source commit: 3faffbb8da00b773cc408a9c52a93ef5aa1c717f Model SHA256: 7fd9e8db551c25c22d12089c64c403fdb66ad99aae1a0f1d2ae34af018d657a6

MODEL: resnet18
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>